# AI Resume Screening and Candidate Ranking System

This notebook shows the complete training workflow used for this project: PDF extraction, NLP preprocessing, TF-IDF training, artifact export, cosine-similarity ranking, and explainable skill-gap detection.

> The match percentage is cosine similarity expressed on a 0–100 scale. It is not a hiring probability and should support human review.

## 1. Install dependencies

In [1]:
%pip install -q "pdfplumber==0.11.7" "Pillow<12" nltk scikit-learn pandas numpy matplotlib seaborn tqdm joblib

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


## 2. Imports and paths
The path resolver works when the notebook is opened from either the project root or `notebooks/`.

In [2]:
from pathlib import Path
from collections import Counter
import logging
import os
import re
import warnings

import joblib
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import pdfplumber
import seaborn as sns
from IPython.display import display
from joblib import Parallel, delayed
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import wordpunct_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger('resume-screening')
sns.set_theme(style='whitegrid', palette='viridis')
pd.set_option('display.max_colwidth', 120)

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'data').is_dir():
            return candidate
    raise FileNotFoundError("Could not find the 'data/data' dataset folder.")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data' / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Dataset: {DATA_DIR}')
print(f'Artifacts: {MODELS_DIR}')

Dataset: c:\Users\vinee\Downloads\FUTURE_ML_03\data\data
Artifacts: c:\Users\vinee\Downloads\FUTURE_ML_03\models


c:\Users\vinee\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Prepare NLTK resources

In [ ]:
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    _create_unverified_https_context = None

if _create_unverified_https_context is not None:
    ssl._create_default_https_context = _create_unverified_https_context


def nltk_resource_exists(resource_path):
    for candidate in (resource_path, f'{resource_path}.zip'):
        try:
            nltk.data.find(candidate)
            return True
        except LookupError:
            continue
    return False


def ensure_nltk_resources():
    resources = {'corpora/stopwords': 'stopwords', 'corpora/wordnet': 'wordnet'}
    for resource_path, package in resources.items():
        if not nltk_resource_exists(resource_path):
            logger.info('Downloading NLTK resource: %s', package)
            try:
                downloaded = nltk.download(package, quiet=True)
            except Exception as exc:
                raise RuntimeError(f'Failed to download NLTK resource: {package}') from exc
            if not downloaded:
                raise RuntimeError(f'Failed to download NLTK resource: {package}')


ensure_nltk_resources()
STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()
print(f'Loaded {len(STOP_WORDS)} English stopwords.')

INFO: Downloading NLTK resource: stopwords
[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1082)>


RuntimeError: Failed to download NLTK resource: stopwords

## 4. Extract every PDF resume
Each PDF is handled independently. Blank, corrupted, or encrypted files are recorded and skipped. Joblib uses up to eight worker processes so the 2,484-file dataset does not take nearly an hour to parse. Results are sorted back into deterministic file order.

In [ ]:
def extract_pdf_text(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text(x_tolerance=2, y_tolerance=2)
            if text:
                pages.append(text)
    return '\n'.join(pages).strip()

def process_pdf(index, pdf_path):
    try:
        text = extract_pdf_text(pdf_path)
        if not text:
            raise ValueError('PDF contains no extractable text')
        record = {
            'resume_name': pdf_path.name,
            'category': pdf_path.parent.name,
            'original_text': text,
        }
        return index, record, None
    except Exception as exc:
        failure = {
            'resume_name': pdf_path.name,
            'path': str(pdf_path),
            'error': f'{type(exc).__name__}: {exc}',
        }
        return index, None, failure

def load_resume_dataset(data_dir, max_workers=None):
    pdf_paths = sorted(data_dir.rglob('*.pdf'))
    if not pdf_paths:
        raise FileNotFoundError(f'No PDF files found under {data_dir}')

    workers = max_workers or min(8, os.cpu_count() or 4)
    results = Parallel(
        n_jobs=workers, backend='loky', return_as='generator_unordered'
    )(delayed(process_pdf)(i, path) for i, path in enumerate(pdf_paths))

    indexed_records, failures = [], []
    for index, record, failure in tqdm(
        results, total=len(pdf_paths), desc='Extracting PDF resumes', unit='pdf'
    ):
        if record is not None:
            indexed_records.append((index, record))
        else:
            failures.append(failure)
            logger.warning('Skipped %s: %s', failure['resume_name'], failure['error'])

    records = [record for _, record in sorted(indexed_records)]
    if not records:
        raise RuntimeError('No usable resume text was extracted.')
    return pd.DataFrame(records), pd.DataFrame(failures), len(pdf_paths)

resumes_df, failed_pdfs_df, discovered_pdf_count = load_resume_dataset(DATA_DIR)
print(f'Total PDFs discovered: {discovered_pdf_count:,}')
print(f'Total resumes processed: {len(resumes_df):,}')
print(f'Number of failed PDFs: {len(failed_pdfs_df):,}')
if not failed_pdfs_df.empty:
    display(failed_pdfs_df)

## 5. NLP preprocessing
Lowercase → remove numbers/punctuation/special characters → normalize spaces → NLTK tokenize → remove stopwords → lemmatize.

In [ ]:
def preprocess_text(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = wordpunct_tokenize(text)
    tokens = [
        LEMMATIZER.lemmatize(token)
        for token in tokens
        if token not in STOP_WORDS and len(token) > 1
    ]
    return ' '.join(tokens)

tqdm.pandas(desc='Preprocessing resumes')
resumes_df['cleaned_text'] = resumes_df['original_text'].progress_apply(preprocess_text)
empty_rows = resumes_df['cleaned_text'].eq('')
if empty_rows.any():
    logger.warning('Dropping %d empty cleaned resumes.', empty_rows.sum())
    resumes_df = resumes_df.loc[~empty_rows].reset_index(drop=True)
resumes_df['original_length'] = resumes_df['original_text'].str.split().str.len()
resumes_df['cleaned_length'] = resumes_df['cleaned_text'].str.split().str.len()
print(f'Dataset shape: {resumes_df.shape}')
display(resumes_df[['resume_name', 'category', 'original_text', 'cleaned_text']].head())

## 6. Train the TF-IDF retrieval index
No classifier is trained. The fitted vectorizer and sparse matrix form the searchable resume index.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    dtype=np.float32,
)
resume_vectors = tfidf_vectorizer.fit_transform(resumes_df['cleaned_text'])
print(f'TF-IDF vocabulary size: {len(tfidf_vectorizer.vocabulary_):,}')
print(f'TF-IDF matrix shape: {resume_vectors.shape}')
print(f'Matrix sparsity: {100 * (1 - resume_vectors.nnz / np.prod(resume_vectors.shape)):.2f}%')

## 7. Training summary and visualizations

In [ ]:
category_distribution = resumes_df['category'].value_counts()
print(f'Total resumes processed: {len(resumes_df):,}')
print(f'Number of failed PDFs: {len(failed_pdfs_df):,}')
print(f'Dataset shape: {resumes_df.shape}')
print(f'TF-IDF vocabulary size: {len(tfidf_vectorizer.vocabulary_):,}')
print(f'TF-IDF matrix shape: {resume_vectors.shape}')
display(category_distribution.rename('resume_count').to_frame())

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
sns.countplot(data=resumes_df, y='category', order=category_distribution.index, ax=axes[0], color='#2a9d8f')
axes[0].set_title('Resume Count by Category')
axes[0].set_xlabel('Number of Resumes')
sns.histplot(resumes_df['original_length'], bins=40, kde=True, ax=axes[1], color='#e76f51')
axes[1].set_title('Distribution of Resume Lengths')
axes[1].set_xlabel('Words per Resume')
plt.tight_layout()
plt.show()

term_counts = Counter()
for document in resumes_df['cleaned_text']:
    term_counts.update(document.split())
top_terms_df = pd.DataFrame(term_counts.most_common(30), columns=['term', 'frequency'])
plt.figure(figsize=(12, 9))
sns.barplot(data=top_terms_df, x='frequency', y='term', color='#457b9d')
plt.title('Top 30 Most Frequent Terms')
plt.tight_layout()
plt.show()
display(top_terms_df)

## 8. Export deployment artifacts

In [ ]:
def save_artifacts(vectorizer, vectors, metadata, output_dir):
    output_dir.mkdir(parents=True, exist_ok=True)
    paths = [
        output_dir / 'tfidf_vectorizer.pkl',
        output_dir / 'resume_vectors.pkl',
        output_dir / 'resume_metadata.csv',
    ]
    try:
        joblib.dump(vectorizer, paths[0], compress=3)
        joblib.dump(vectors, paths[1], compress=3)
        metadata.to_csv(paths[2], index=False, encoding='utf-8')
    except OSError as exc:
        raise RuntimeError(f'Could not save artifacts: {exc}') from exc
    if any(not path.exists() or path.stat().st_size == 0 for path in paths):
        raise RuntimeError('Artifact verification failed.')
    return paths

metadata_columns = ['resume_name', 'category', 'original_text', 'cleaned_text']
artifact_paths = save_artifacts(
    tfidf_vectorizer, resume_vectors, resumes_df[metadata_columns], MODELS_DIR
)
for path in artifact_paths:
    print(f'Saved: {path.name} ({path.stat().st_size / 1_048_576:.2f} MB)')

## 9. Candidate ranking and skill-gap demo
The fitted vectorizer transforms a job description, cosine similarity ranks every resume, and a transparent skill catalogue explains matches and gaps.

In [ ]:
SKILL_CATALOG = (
    'accounting', 'auditing', 'budgeting', 'financial analysis', 'financial reporting',
    'python', 'java', 'javascript', 'sql', 'html', 'css', 'react', 'django', 'flask',
    'machine learning', 'deep learning', 'natural language processing', 'computer vision',
    'data analysis', 'data visualization', 'statistics', 'pandas', 'numpy', 'scikit-learn',
    'tensorflow', 'pytorch', 'power bi', 'tableau', 'database management',
    'aws', 'azure', 'google cloud', 'cloud computing', 'docker', 'kubernetes', 'linux',
    'git', 'devops', 'api development', 'rest api', 'microservices', 'cybersecurity',
    'project management', 'product management', 'agile', 'scrum', 'business analysis',
    'sales', 'customer service', 'marketing', 'digital marketing', 'seo', 'public relations',
    'recruitment', 'talent acquisition', 'onboarding', 'payroll', 'human resources',
    'graphic design', 'ui design', 'ux design', 'autocad', 'construction management',
    'patient care', 'healthcare management', 'teaching', 'curriculum development',
    'legal research', 'legal writing', 'litigation', 'contract management', 'compliance',
    'communication', 'leadership', 'teamwork', 'problem solving', 'time management',
)
NORMALIZED_SKILLS = {skill: preprocess_text(skill) for skill in SKILL_CATALOG}

def term_is_present(term, cleaned_text):
    return re.search(rf'(?<!\w){re.escape(term)}(?!\w)', cleaned_text) is not None

def extract_skills(text):
    cleaned = preprocess_text(text)
    return [
        skill for skill, normalized in NORMALIZED_SKILLS.items()
        if normalized and term_is_present(normalized, cleaned)
    ]

def rank_candidates(job_description, vectorizer, vectors, metadata, top_k=10):
    if not isinstance(job_description, str) or not job_description.strip():
        raise ValueError('Job description must be a non-empty string.')
    if len(metadata) != vectors.shape[0]:
        raise ValueError('Metadata and resume-vector rows are misaligned.')

    cleaned_job = preprocess_text(job_description)
    job_vector = vectorizer.transform([cleaned_job])
    if job_vector.nnz == 0:
        raise ValueError('Job description has no terms in the trained vocabulary.')
    scores = cosine_similarity(job_vector, vectors).ravel()
    ranked_indices = np.argsort(scores)[::-1][:min(top_k, len(scores))]
    required_skills = extract_skills(job_description)

    rows = []
    for rank, idx in enumerate(ranked_indices, start=1):
        resume_text = metadata.iloc[idx]['cleaned_text']
        matched = [s for s in required_skills if term_is_present(preprocess_text(s), resume_text)]
        missing = [s for s in required_skills if not term_is_present(preprocess_text(s), resume_text)]
        rows.append({
            'rank': rank,
            'resume_name': metadata.iloc[idx]['resume_name'],
            'category': metadata.iloc[idx]['category'],
            'match_percentage': round(float(scores[idx]) * 100, 2),
            'matched_skills': ', '.join(matched) if matched else 'None detected',
            'skill_gaps': ', '.join(missing) if missing else 'No major gaps detected',
        })
    return pd.DataFrame(rows)

sample_job_description = (
    'We need a software engineer with Python, SQL, machine learning, data analysis, Git, '
    'cloud computing, API development, communication, and project management skills.'
)
display(rank_candidates(
    sample_job_description, tfidf_vectorizer, resume_vectors, resumes_df, top_k=10
))

## 10. Final verification

In [ ]:
assert resume_vectors.shape[0] == len(resumes_df)
assert len(tfidf_vectorizer.vocabulary_) == resume_vectors.shape[1]
assert all(path.exists() and path.stat().st_size > 0 for path in artifact_paths)
print('Training completed successfully.')
print('Artifacts saved in models/.')